# Completing Random Forest Analysis
Assumption that appropriate separation and PCA has been completed.

## SETUP

In [18]:
import subprocess, sys, os, importlib, re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import rf_functions 
importlib.reload(rf_functions)

<module 'rf_functions' from 'c:\\Users\\jespe\\Desktop\\[THESIS] Code\\-ESPEJO-THESIS-DOS-ML-Model-\\SCRIPTS_RANDOM_FOREST\\rf_functions.py'>

In [25]:
# STEP 0 : LOAD THE ADSORPTION ENERGY DATA

def overall_rf(transformed_data_dir, dataset):
    adsorption_energy_df = pd.read_excel('Adsorption_energies_additives.xlsx')

    # Initialize an empty list to store the results
    results = []
    
    files = [f for f in os.listdir(transformed_data_dir) if f.startswith('transformed')]

    for filename in files:
        filepath = os.path.join(transformed_data_dir, filename)

        # Load and process the data
        pca_data = rf_functions.load_pca_transformed_data(filepath)
        merged_data = rf_functions.adsorption_energy_for_dataset(pca_data, adsorption_energy_df)

        # Perform Random Forest and get mse and r2
        mae, mse, rmse, r2, adjusted_r2 = rf_functions.perform_random_forest(merged_data)

        # Extract the energy range and cumulative variance ratio from the filename
        energy_range_match = re.search(r'\[([-+]?\d*\.\d+|\d+),([-+]?\d*\.\d+|\d+)\]', filename)
        cum_variance_match = re.search(r'var([\d\.]+)', filename)

        if energy_range_match and cum_variance_match:
            energy_range = f"{energy_range_match.group(1)},{energy_range_match.group(2)}"
            cum_variance = cum_variance_match.group(1).rstrip('.')

            # Append the results to the list as a dictionary
            results.append({
                'dataset': dataset.replace('transformed_data_', ''),
                'energy_range': energy_range,
                'cum_variance': cum_variance,
                'n_pc': pca_data.shape[1] - 1,
                # 'mean_absolute_error': mae,
                'mean_squared_error': mse,
                #'root_mean_squared_error': rmse,
                'r_squared': r2,
                #'adjusted_r2': adjusted_r2
            })

    # Convert the results list into a DataFrame
    results_df = pd.DataFrame(results)

    # Define the output CSV file path
    output_path = os.path.join(transformed_data_dir, 'rf_results.txt')

    # Save the DataFrame to a CSV file
    results_df.to_csv(output_path, sep='\t', index=False)

    return results_df

In [20]:
# Create an empty DataFrame
overall_results = pd.DataFrame(columns=['dataset',
                                        'energy_range',
                                        'cum_variance',
                                        'n_pc',
                                        'mean_absolute_error',
                                        'mean_squared_error',
                                        'root_mean_squared_error',
                                        'r_squared',
                                        'adjusted_r2'], dtype='float64')

overall_results

,dataset,energy_range,cum_variance,n_pc,mean_absolute_error,mean_squared_error,root_mean_squared_error,r_squared,adjusted_r2


In [21]:
base_dir = os.getcwd()

transformed_data_dirs = [f for f in os.listdir(base_dir) if f.startswith('transformed_data')]

for dir in transformed_data_dirs:
    print(f"Processing directory is {dir}")
    dir_path = os.path.join(base_dir, dir)
    cur_results = overall_rf(dir_path, dir)
    overall_results = pd.concat([overall_results, cur_results], ignore_index=True).drop_duplicates()

overall_results

Processing directory is transformed_data_DOSCAR_files_1
Processing directory is transformed_data_DOSCAR_files_1-2
Processing directory is transformed_data_DOSCAR_files_1-3
Processing directory is transformed_data_DOSCAR_files_1-4
Processing directory is transformed_data_DOSCAR_files_1-5


,dataset,energy_range,cum_variance,n_pc,mean_absolute_error,mean_squared_error,root_mean_squared_error,r_squared,adjusted_r2
0,DOSCAR_files_1,"-10.00,10.00",0.80,11.0,0.188212,0.047958,0.218993,-0.271306,-3.068180
1,DOSCAR_files_1,"-10.00,10.00",0.90,22.0,0.194571,0.047335,0.217567,-0.254805,4.346148
2,DOSCAR_files_1,"-5.00,5.00",0.80,8.0,0.219382,0.068105,0.260970,-0.805396,-2.610792
3,DOSCAR_files_1,"-5.00,5.00",0.90,15.0,0.250165,0.078869,0.280835,-1.090718,-32.451485
4,DOSCAR_files_1-2,"-10.00,10.00",0.80,14.0,0.171711,0.055645,0.235892,0.095254,-0.960283
5,DOSCAR_files_1-2,"-10.00,10.00",0.90,28.0,0.171141,0.059643,0.244219,0.030258,13.606646
6,DOSCAR_files_1-2,"-5.00,5.00",0.80,10.0,0.205941,0.067026,0.258894,-0.089790,-0.770908
7,DOSCAR_files_1-2,"-5.00,5.00",0.90,19.0,0.185193,0.066869,0.258591,-0.087239,-3.038315
8,DOSCAR_files_1-3,"-10.00,10.00",0.80,35.0,0.189508,0.063222,0.251440,-0.005018,3.192767
9,DOSCAR_files_1-3,"-10.00,10.00",0.90,50.0,0.192344,0.062937,0.250873,-0.000485,1.923525


## FOR DATASET 1

In [30]:
transformed_data_dir = os.path.join(os.getcwd(),'transformed_data_DOSCAR_files_1')

cur_results_set_1 = overall_rf(transformed_data_dir, 'set_1')
cur_results_set_1 

,dataset,energy_range,cum_variance,n_pc,mean_squared_error,r_squared
0,set_1,"-10.00,10.00",0.80,11,0.046905,-0.243396
1,set_1,"-10.00,10.00",0.90,22,0.049902,-0.322852
2,set_1,"-11.44,4.12",0.80,13,0.051453,-0.363953
3,set_1,"-11.44,4.12",0.90,24,0.050303,-0.333464
4,set_1,"-5.00,5.00",0.80,8,0.068108,-0.805458
5,set_1,"-5.00,5.00",0.90,15,0.072142,-0.912413


## FOR DATASET 1-5

In [31]:
transformed_data_dir = os.path.join(os.getcwd(),'transformed_data_DOSCAR_files_1-5')

cur_results_set_5 = overall_rf(transformed_data_dir, 'set 1-5')
cur_results_set_5

,dataset,energy_range,cum_variance,n_pc,mean_squared_error,r_squared
0,set 1-5,"-1.46,3.27",0.80,10,0.046603,0.113513
1,set 1-5,"-1.46,3.27",0.90,14,0.042917,0.183625
2,set 1-5,"-10.00,10.00",0.80,39,0.040639,0.226965
3,set 1-5,"-10.00,10.00",0.90,57,0.046272,0.119804
4,set 1-5,"-5.00,5.00",0.80,25,0.037570,0.285343
5,set 1-5,"-5.00,5.00",0.90,36,0.040797,0.223943


In [32]:
combined_df = pd.concat([cur_results_set_1, cur_results_set_5], ignore_index=True).drop_duplicates()
combined_df

,dataset,energy_range,cum_variance,n_pc,mean_squared_error,r_squared
0,set_1,"-10.00,10.00",0.80,11,0.046905,-0.243396
1,set_1,"-10.00,10.00",0.90,22,0.049902,-0.322852
2,set_1,"-11.44,4.12",0.80,13,0.051453,-0.363953
3,set_1,"-11.44,4.12",0.90,24,0.050303,-0.333464
4,set_1,"-5.00,5.00",0.80,8,0.068108,-0.805458
5,set_1,"-5.00,5.00",0.90,15,0.072142,-0.912413
6,set 1-5,"-1.46,3.27",0.80,10,0.046603,0.113513
7,set 1-5,"-1.46,3.27",0.90,14,0.042917,0.183625
8,set 1-5,"-10.00,10.00",0.80,39,0.040639,0.226965
9,set 1-5,"-10.00,10.00",0.90,57,0.046272,0.119804
